# Lesion Analysis


In [11]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
import umap
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances

/home/etosato/miniconda3/envs/nemesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_ROOT = Path("/home/etosato/Projects/NEMESIS_fs/data/clinical_connectome")
DATASETS = ["UNIPD/WashU", "UNIPD/PASPORT", "UNIPD/PSP", "UKLFR/stroke_UKLFR"]
LESION_GLOB = "*/lesion/mni/*_label-lesion_mask.nii.gz"
CACHE_DIR = Path("/home/etosato/Projects/NEMESIS_fs/data/derived/lesion_matrix")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

Cerca tutte le lesion mask MNI per i 4 dataset in data/clinical_connectome, verificando che ogni cartella soggetto ne contenga esattamente una (nessun soggetto mancante o duplicato) e che i file trovati siano tutti .nii.gz. Stampa il conteggio per dataset e il totale.

In [6]:
lesion_files: dict[str, list[Path]] = {}

for dataset in DATASETS:
    files = sorted((DATA_ROOT / dataset).glob(LESION_GLOB))

    # subject dirs only — excludes root-level files like participants.tsv
    n_subject_dirs = len([p for p in (DATA_ROOT / dataset).iterdir() if p.is_dir()])

    # one lesion mask expected per subject dir; stop on mismatch rather than
    # silently proceeding with missing or duplicated data
    if len(files) != n_subject_dirs:
        raise ValueError(
            f"{dataset}: {n_subject_dirs} subject dirs but only "
            f"{len(files)} lesion masks found — check the retrieval report"
        )

    lesion_files[dataset] = files
    print(f"{dataset:25s} {len(files):4d} lesions")

total = sum(len(files) for files in lesion_files.values())
print(f"{'TOTAL':25s} {total:4d} lesions")


UNIPD/WashU                202 lesions
UNIPD/PASPORT               83 lesions
UNIPD/PSP                  168 lesions
UKLFR/stroke_UKLFR         697 lesions
TOTAL                     1150 lesions


Carica una lesione di riferimento (WashU) per fissare la griglia voxel comune 

- **91×109×91**: numero di voxel lungo i tre assi (x, y, z) della griglia, cioè la shape dell'array 3D
- **@ 2mm**: dimensione di ciascun voxel lungo ogni asse. Moltiplicando shape × zoom ottieni l'estensione fisica reale: 91×2=182mm, 109×2=218mm, 91×2=182mm.

- **MNI152NLin6Asym**:  nome del template/spazio di coordinate a cui l'immagine è registrata "Montreal Neurological Institute 152, versione NonLinear 6th generation, Asymmetric". È lo standard usato da FSL. Indica dove sta l'immagine nello spazio cerebrale standard, non quanto è densa la griglia (due immagini possono essere nello stesso spazio ma con risoluzioni diverse)
 
a cui verranno riportate tutte le altre lesioni con risoluzione diversa (es: 1mm PASPORT/PSP, 1.5mm UKLFR).

In [7]:
reference_img = nib.load(lesion_files["UNIPD/WashU"][0])
print("reference grid:", reference_img.shape, reference_img.header.get_zooms())

reference grid: (91, 109, 91) (np.float32(2.0), np.float32(2.0), np.float32(2.0))


Carica ogni lesion mask, la ricampiona sulla griglia comune (91×109×91 @ 2mm) quando la risoluzione nativa è diversa, la ribinarizza e la appiattisce in un vettore. 

Impila tutti i vettori in una matrice X (n_soggetti × n_voxel) e costruisce metadata con soggetto e dataset di origine, mantenendo lo stesso ordine di riga per riga tra X e metadata.

In [5]:
subject_ids: list[str] = []
dataset_labels: list[str] = []
vectors: list[np.ndarray] = []

for dataset, files in lesion_files.items():
    for f in files:
        img = nib.load(f)
        # resample onto the common grid when resolution differs;
        # nearest-neighbor keeps the mask binary
        if img.shape != reference_img.shape:
            img = resample_to_img(
                img, reference_img, interpolation="nearest", force_resample=True, copy_header=True
            )
        # re-binarize: interpolation/registration can leave near-1/near-0 values
        data = img.get_fdata() > 0.5
        # flatten to a 1D vector and convert to uint8 for memory efficiency
        vectors.append(data.ravel().astype(np.uint8))
        subject_ids.append(f.name.split("_")[0])
        dataset_labels.append(dataset)

# stack the list of 1D vectors into a 2D matrix: shape (n_subjects, n_voxels)
X = np.stack(vectors)
# metadataframe has the same order as the rows of X
metadata = pd.DataFrame({"subject_id": subject_ids, "dataset": dataset_labels})
print("X shape:", X.shape)
metadata["dataset"].value_counts()

X shape: (1150, 902629)


dataset
UKLFR/stroke_UKLFR    697
UNIPD/WashU           202
UNIPD/PSP             168
UNIPD/PASPORT          83
Name: count, dtype: int64

Rimuove i voxel mai lesionati in nessun soggetto (colonne costanti a 0). Non ha alcun effetto su t-SNE/UMAP — una colonna costante non contribuisce a nessuna distanza tra soggetti — è solo un'ottimizzazione di memoria e velocità.

In [ ]:
# true for voxels lesioned in at least one subject (any() reduces along axis=0, the subjects)
non_constant = X.any(axis=0)
X_reduced = X[:, non_constant]
print(f"voxel totali: {X.shape[1]}, non costanti (usati): {X_reduced.shape[1]}")

voxel totali: 902629, non costanti (usati): 254865


In [ ]:
# Save to .npy
np.save(CACHE_DIR / "X_reduced.npy", X_reduced)
np.save(CACHE_DIR / "non_constant.npy", non_constant)
metadata.to_csv(CACHE_DIR / "metadata.csv", index=False)

print(f"saved in {CACHE_DIR}/")

In [8]:
# Load from .npy
X_reduced = np.load(CACHE_DIR / "X_reduced.npy")
non_constant = np.load(CACHE_DIR / "non_constant.npy")
metadata = pd.read_csv(CACHE_DIR / "metadata.csv")
print("X_reduced shape:", X_reduced.shape)

X_reduced shape: (1150, 254865)
